# Low-level data reduction using ctapipe

### Check paths and environments

In [1]:
# Where are we?
! pwd


/Users/tmiener/deeplearning/iaa-advanced-neural-networks-2026/day3_ctlearn_gammaray/notebooks


In [2]:
# Check for the versions of the core dependencies 
! conda list | grep ctlearn
! conda list | grep astropy
! conda list | grep ctapipe
! conda list | grep dl1-data-handler 
! conda list | grep tensorflow
! conda list | grep keras

# packages in environment at /Users/tmiener/miniforge3/envs/ctlearn:
ctlearn                          0.10.3               pypi_0              pypi
astropy                          8.0.0                pypi_0              pypi
astropy-iers-data                0.2026.6.22.1.23.34  pypi_0              pypi
ctapipe                          0.32.0               pypi_0              pypi
dl1-data-handler                 0.14.9               pypi_0              pypi
tensorflow                       2.21.0               pypi_0              pypi
keras                            3.15.0               pypi_0              pypi


In [3]:
# Set here correct path to downloaded test data of CTAO simulation.
DATA_DIR = "../../../iaa-advanced-neural-networks-2026-ctao-data"
! du -h {DATA_DIR}/simtel/*/*

435M	../../../iaa-advanced-neural-networks-2026-ctao-data/simtel/gamma-diffuse/simtel_corsika_theta_16.087_az_108.090_run1.simtel.gz
426M	../../../iaa-advanced-neural-networks-2026-ctao-data/simtel/gamma-diffuse/simtel_corsika_theta_16.087_az_108.090_run2.simtel.gz
449M	../../../iaa-advanced-neural-networks-2026-ctao-data/simtel/gamma-diffuse/simtel_corsika_theta_16.087_az_108.090_run3.simtel.gz
452M	../../../iaa-advanced-neural-networks-2026-ctao-data/simtel/gamma-diffuse/simtel_corsika_theta_16.087_az_108.090_run4.simtel.gz
 81M	../../../iaa-advanced-neural-networks-2026-ctao-data/simtel/proton/simtel_corsika_theta_16.087_az_108.090_run1.simtel.gz
 71M	../../../iaa-advanced-neural-networks-2026-ctao-data/simtel/proton/simtel_corsika_theta_16.087_az_108.090_run2.simtel.gz
 68M	../../../iaa-advanced-neural-networks-2026-ctao-data/simtel/proton/simtel_corsika_theta_16.087_az_108.090_run3.simtel.gz
 74M	../../../iaa-advanced-neural-networks-2026-ctao-data/simtel/proton/simtel_corsika_the

In [4]:
# Create directories to store processed data products 
!mkdir -p {DATA_DIR}/hdf5
!mkdir -p {DATA_DIR}/hdf5_merged

In [5]:
# Exploring the ctapipe tools we are going to use in the background:
! ctapipe-process -h
! ctapipe-merge -h

Process data from lower-data levels up to DL1 and DL2, including image
extraction and optionally image parameterization as well as muon analysis and
shower reconstruction.

Note that the muon analysis and shower reconstruction both depend on
parametrized images and therefore compute image parameters even if
DataWriter.write_dl1_parameters=False in case these are not already present
in the input file.
 This currently uses data model version v7.6.0

Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    Set log-level to debug, for the most verbose logging.
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
 

### Reducing the data
The frist step is the reduction of simtel files to HDF5 files following the official CTAO data model. Several data levels can be stored within one file. As an example, we are storing here the calibrated waveforms (R1) and the integrated images (DL1a) and image parameters (DL1b). This operation is done run-wise and can be easily parallelize on a cluster using sbatch. An additional script can be found in "../scripts/run_reduction_cluster.py" for the CTAO onsite data center. The scripts used for this reduction needs to be adjusted according to the naming of the simtel file. It is recommended carefully check before a major reduction is processed.

In [6]:
! python ../scripts/run_reduction.py \
--input_dir {DATA_DIR}/simtel/gamma-diffuse/ \
--type gamma \
--config ../configs/ctapipe_standard_LST1_config.json \
--output_dir {DATA_DIR}/hdf5/ \
--log_level INFO

Inputfile: '/Users/tmiener/deeplearning/iaa-advanced-neural-networks-2026-ctao-data/simtel/gamma-diffuse/simtel_corsika_theta_16.087_az_108.090_run1.simtel.gz'
Outputfile: '../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5//gamma_theta_16.087_az_108.090_run1.r1.dl1.h5'
2026-09-22 13:04:48,683 INFO [ctapipe.ctapipe-process] (tool.initialize): Loading config from '[PosixPath('/Users/tmiener/deeplearning/iaa-advanced-neural-networks-2026/day3_ctlearn_gammaray/notebooks/../configs/ctapipe_standard_LST1_config.json')]'
2026-09-22 13:04:48,684 INFO [ctapipe.ctapipe-process] (tool.initialize): ctapipe version 0.32.0
2026-09-22 13:04:48,685 INFO [ctapipe.ctapipe-process.SimTelEventSource] (eventsource.__init__): INPUT PATH = /Users/tmiener/deeplearning/iaa-advanced-neural-networks-2026-ctao-data/simtel/gamma-diffuse/simtel_corsika_theta_16.087_az_108.090_run1.simtel.gz
2026-09-22 13:04:48,734 INFO [ctapipe.ctapipe-process] (process.start): applying calibration: True
2026-09-22 13:04:48

We are also reducing a couple of proton files.

In [7]:
! python ../scripts/run_reduction.py \
--input_dir {DATA_DIR}/simtel/proton/ \
--type proton \
--config ../configs/ctapipe_standard_LST1_config.json \
--output_dir {DATA_DIR}/hdf5/ \
--log_level ERROR

Inputfile: '/Users/tmiener/deeplearning/iaa-advanced-neural-networks-2026-ctao-data/simtel/proton/simtel_corsika_theta_16.087_az_108.090_run1.simtel.gz'
Outputfile: '../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5//proton_theta_16.087_az_108.090_run1.r1.dl1.h5'
Inputfile: '/Users/tmiener/deeplearning/iaa-advanced-neural-networks-2026-ctao-data/simtel/proton/simtel_corsika_theta_16.087_az_108.090_run2.simtel.gz'
Outputfile: '../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5//proton_theta_16.087_az_108.090_run2.r1.dl1.h5'
Inputfile: '/Users/tmiener/deeplearning/iaa-advanced-neural-networks-2026-ctao-data/simtel/proton/simtel_corsika_theta_16.087_az_108.090_run3.simtel.gz'
Outputfile: '../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5//proton_theta_16.087_az_108.090_run3.r1.dl1.h5'
Inputfile: '/Users/tmiener/deeplearning/iaa-advanced-neural-networks-2026-ctao-data/simtel/proton/simtel_corsika_theta_16.087_az_108.090_run4.simtel.gz'
Outputfile: '../../../iaa-advan

After production we can check the files in the output folder.

In [8]:
! du -h {DATA_DIR}/hdf5/*

640M	../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5/gamma_theta_16.087_az_108.090_run1.r1.dl1.h5
625M	../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5/gamma_theta_16.087_az_108.090_run2.r1.dl1.h5
657M	../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5/gamma_theta_16.087_az_108.090_run3.r1.dl1.h5
657M	../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5/gamma_theta_16.087_az_108.090_run4.r1.dl1.h5
 97M	../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5/proton_theta_16.087_az_108.090_run1.r1.dl1.h5
 80M	../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5/proton_theta_16.087_az_108.090_run2.r1.dl1.h5
 80M	../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5/proton_theta_16.087_az_108.090_run3.r1.dl1.h5
 81M	../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5/proton_theta_16.087_az_108.090_run4.r1.dl1.h5
 96M	../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5/proton_theta_16.087_az_108.090_run5.r1.dl1.h5
 80M	../../../iaa-advan

### Merging the HDF5 files
It is HIGHLY RECOMMENDED to merge the HDF5 files. Usually, the whole production is merged into 100 files, where we split 80 files into the training and validation set and reserve 20 files for the test set to build the instrumental response functions (IRFs). Working with merged files ensure a rather smooth handling of the training process with DL1DH+CTLearn.

In [9]:
! python ../scripts/run_merger.py \
--input_dir {DATA_DIR}/hdf5/ \
--pattern "gamma*" \
--num_outputfiles 2 \
--output_dir {DATA_DIR}/hdf5_merged/ \
--log_level INFO

Outputfile: '../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5_merged//gamma_theta_16.087_az_108.090_runs1-2.r1.dl1.h5'
2026-09-22 13:15:03,728 INFO [ctapipe.ctapipe-merge] (tool.initialize): Loading config from '[]'
2026-09-22 13:15:03,729 INFO [ctapipe.ctapipe-merge] (tool.initialize): ctapipe version 0.32.0
2026-09-22 13:16:54,955 INFO [ctapipe.ctapipe-merge.HDF5Merger] (hdf5merger._get_required_nodes): Updated required nodes to ['/configuration/observation/observation_block', '/configuration/observation/scheduling_block', '/configuration/simulation/run', '/configuration/telescope/pointing', '/dl1/event/subarray/trigger', '/dl1/event/telescope/images', '/dl1/event/telescope/parameters', '/dl1/event/telescope/trigger', '/r1/event/telescope', '/simulation', '/simulation/event/subarray/shower', '/simulation/event/telescope/images', '/simulation/event/telescope/impact', '/simulation/event/telescope/parameters', '/simulation/service/shower_distribution']
2026-09-22 13:18:47,048 I

In [10]:
! python ../scripts/run_merger.py \
--input_dir {DATA_DIR}/hdf5/ \
--pattern "proton*" \
--num_outputfiles 3 \
--output_dir {DATA_DIR}/hdf5_merged/ \
--log_level ERROR

Outputfile: '../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5_merged//proton_theta_16.087_az_108.090_runs1-2.r1.dl1.h5'
Outputfile: '../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5_merged//proton_theta_16.087_az_108.090_runs3-4.r1.dl1.h5'
Outputfile: '../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5_merged//proton_theta_16.087_az_108.090_runs5-6.r1.dl1.h5'


In [11]:
! du -h {DATA_DIR}/hdf5_merged/*

1.2G	../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5_merged/gamma_theta_16.087_az_108.090_runs1-2.r1.dl1.h5
1.3G	../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5_merged/gamma_theta_16.087_az_108.090_runs3-4.r1.dl1.h5
176M	../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5_merged/proton_theta_16.087_az_108.090_runs1-2.r1.dl1.h5
160M	../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5_merged/proton_theta_16.087_az_108.090_runs3-4.r1.dl1.h5
160M	../../../iaa-advanced-neural-networks-2026-ctao-data/hdf5_merged/proton_theta_16.087_az_108.090_runs5-6.r1.dl1.h5


### Browsing through the HDF5 files via vitables
Use ViTables, a convenient GUI, to explore the data model.

In [12]:
#! conda run -n vitables vitables {DATA_DIR}/hdf5_merged/*